In [1]:
import sys, pathlib as pl; sys.path.insert(0, str(pl.Path.cwd().parents[1])); from fig_utils import fig_csv
import numpy as np
import pandas as pd
import matplotlib as mpl
mpl.use('Cairo')  # for saving SVGs that Affinity Designer can parse
import matplotlib.pyplot as plt
import seaborn as sns
import dill

from candas.utils import setup_paths
from candas.style import breve
from candas.learn import ParameterSet
import gumbi as gmb
import pathlib as pl

code_pth = pl.Path.cwd()  # for running in Jupyter
# code_pth = pl.Path(__file__)  # for running in terminal
fig_pth = code_pth.parent
data_pth = fig_pth / 'data'
graph_pth = fig_pth / 'graphics'
graph_pth.mkdir(exist_ok=True)

gen_pth = fig_pth / 'generated'
gen_pth.mkdir(exist_ok=True)
plt.style.use(str(breve))

%config InlineBackend.figure_format = 'retina'

WARNING (theano.link.c.cmodule): install mkl with `conda install mkl-service`: No module named 'mkl'
/opt/anaconda3/envs/can_manuscript/lib/python3.9/site-packages/arviz/data/base.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
from utils import savefig

In [3]:
with open(gen_pth / "Model_Avg_predictions.pkl", "rb") as f:
    avg_model_r = dill.load(f)["r"]

In [4]:
ps = ParameterSet.load(data_pth / "ADVI_ParameterSets_220528.pkl")


def make_pair(row):
    return "-".join(sorted([row.FPrimer, row.RPrimer]))


data = (
    ps.wide.query('Metric == "mean"')
    .astype({"BP": float})
    .assign(PrimerPair=lambda df: df.apply(make_pair, axis=1))
    .groupby(["Target", "PrimerPair", "Reporter"])
    .mean()
    .reset_index()
)

ds = gmb.DataSet(
    data=data,
    outputs=["F0_lg", "r", "K", "m"],
    log_vars=["BP", "K", "m", "r"],
    logit_vars=["GC"],
)

selected = (
    data.groupby(["PrimerPair", "Reporter"])
    .size()
    .reset_index()
    .rename(columns={0: "Observations"})
    .sort_values("Observations", ascending=False)
    .reset_index(drop=True)
).iloc[[0, 1, 4, 5, 6, 8, 38, 39, 42]]

/var/folders/kb/z3p7xb5x1_191lkf4wxkt1h00000gn/T/ipykernel_35253/3371070553.py:9: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  ps.wide.query('Metric == "mean"')


In [5]:
with open(gen_pth / "Model_LMC_predictions.pkl", "rb") as f:
    predictions_dict = dill.load(f)

all_r = [predictions_dict[f"r{i}"] for i in range(9)]
BP = predictions_dict["BP"]
GC = predictions_dict["GC"]

limits = BP.parray(GC=[0.2, 0.8], BP=[10, 600])

In [9]:
width = 3.45
height = 3.31
figsize = (width, height)
spotsize = 20
linewidth = 2
ticklabelsize = 8
labelsize = 10
titlesize = labelsize + 2

# Set rcParams for plotting
mpl.rc("xtick", labelsize=ticklabelsize)
mpl.rc("ytick", labelsize=ticklabelsize)
mpl.rc("axes", labelsize=labelsize, titlesize=titlesize, linewidth=1)

fig, axs = plt.subplots(3, 3, figsize=figsize, sharey=True, sharex=True)

rnorm = mpl.colors.Normalize(vmin=0.23, vmax=1.03)

for i, (r, ax, row) in enumerate(zip(all_r, axs.flat, selected.itertuples())):
    plt.sca(ax)
    pp = gmb.ParrayPlotter(
        x=GC,
        y=BP,
        z=r,
        #    x_scale='standardized',
        y_scale="standardized",
    )

    step = 0.05
    levels = (
        np.arange(np.floor(rnorm.vmin / step), np.ceil(rnorm.vmax / step) + 1) * step
    )
    cs = pp(plt.contourf, levels=levels, cmap="flare_r", norm=rnorm)
    # pp.colorbar(cs)

    yticks = BP.parray(BP=[10, 30, 100, 300])
    ax.set_yticks(yticks["BP"].z.values())
    ax.set_yticklabels(map(int, yticks.values()))

    xticks = GC.parray(GC=[0.25, 0.5, 0.75])
    ax.set_xticks(xticks["GC"].values())
    ax.set_xticklabels(map(int, 100 * xticks.values()))

    # if i%3 != 0:
    #     ax.set_ylabel('')
    # if i//3 != 2:
    #     ax.set_xlabel('')
    ax.set_ylabel("")
    ax.set_xlabel("")

    gc = ds.wide.query("Reporter == @row.Reporter and PrimerPair == @row.PrimerPair").GC
    bp = ds.wide.z.query(
        "Reporter == @row.Reporter and PrimerPair == @row.PrimerPair"
    ).BP
    rs = ds.wide.query("Reporter == @row.Reporter and PrimerPair == @row.PrimerPair").r

    ds.wide.to_csv(fig_csv('silk_viper'))
    ax.scatter(
        gc,
        bp,
        #    c='0.5', s=1,
        c=rs,
        edgecolor="0.8",
        linewidths=0.5,
        s=spotsize,
        cmap="flare_r",
        norm=rnorm,
    )

    print(gc)
    print(bp)

    ax.set_xlim([0.2, 0.8])

    cs = ax.contour(
        GC.values(),
        BP.z.values(),
        r.σ,
        levels=[0.05, 0.10, 0.15, 0.20, 0.25],
        colors="0.2",
        linestyles="--",
        linewidths=1,
    )
    ax.clabel(cs, fontsize=ticklabelsize)
    print(GC.values())
    print(BP.values())


axs[1, 0].set_ylabel("Length (bp)")
axs[-1, 1].set_xlabel("GC content (%)")


mar_l = 0.55
mar_r = 0.1
mar_t = 0.22
mar_b = 0.48

plt.subplots_adjust(
    left=mar_l / width,
    right=1 - mar_r / width,
    top=1 - mar_t / height,
    bottom=mar_b / height,
)

#savefig(fig, alias="silk_viper")

29     0.479592
30     0.435644
31     0.405660
35     0.250000
37     0.352273
39     0.431818
43     0.545455
45     0.647727
47     0.750000
49     0.795455
51     0.852273
53     0.131250
55     0.466667
57     0.218750
59     0.431250
61     0.587500
63     0.781250
65     0.425000
67     0.420833
69     0.325000
70     0.682143
72     0.418000
74     0.570000
77     0.450000
79     0.260000
81     0.620000
83     0.740000
85     0.454545
87     0.147727
209    0.404255
210    0.320513
211    0.363636
212    0.491525
213    0.259259
214    0.544118
215    0.354331
216    0.460526
Name: GC, dtype: float64
29     0.084571
30     0.127434
31     0.196120
35    -0.068427
37    -0.068427
39    -0.068427
43    -0.068427
45    -0.068427
47    -0.068427
49    -0.068427
51    -0.068427
53     0.781406
55    -1.598173
57     0.781406
59     0.781406
61     0.781406
63     0.781406
65     1.098608
67     1.357780
69     1.576907
70     1.576907
72     2.401127
74     2.401127
77    -1.189230